# ex03 · nb01 — 팔(arm) 모션 총집합 (moveit_py 인터랙티브)

mock Piper 팔을 **노트북 셀로 실시간 조종**하는 예제다. 다양한 종류의 팔 모션을
하나씩 셀 실행으로 돌려본다: SRDF 이름 상태 · 관절공간 목표 · 포즈/IK 목표 · 속도 스케일링.

**먼저 스택을 띄워야 한다** (다른 터미널):

```bash
source /opt/ros/jazzy/setup.bash
source ~/piper-rwh/ros2_ws/install/setup.bash
LC_NUMERIC=C ros2 launch ~/piper-rwh/examples/python/ex03_moveit_py_notebook.launch.py
```
컨트롤러가 전부 active 되고 RViz 에 로봇이 뜨면(~30초) 아래 셀부터 순서대로 **Shift+Enter**.

> ⚠️ **설정 셀(다음 `MoveItPy(...)` 셀)은 커널당 딱 한 번만 실행.** 다시 실행하면 커널이
> 죽을 수 있다(알려진 MoveItPy 소멸자 이슈). 한 번에 **한 노트북/커널만** 컨트롤러를 구동할 것.

In [1]:
import time

import numpy as np
from geometry_msgs.msg import PoseStamped
from moveit.core.kinematic_constraints import construct_joint_constraint
from moveit.core.robot_state import RobotState
from moveit.planning import MoveItPy, PlanRequestParameters
from rclpy.logging import get_logger
import rclpy

from moveit_py_params import build_moveit_config_dict

if not rclpy.ok():
    rclpy.init()  # 로깅용 — MoveItPy 는 자체 백그라운드 스레드에서 spin 한다

logger = get_logger("nb01_arm_motions")
robot = MoveItPy(node_name="moveit_py", config_dict=build_moveit_config_dict())
arm = robot.get_planning_component("arm")
robot_model = robot.get_robot_model()
print("MoveItPy 준비 완료 — 아래 셀들을 하나씩 Shift+Enter")

MoveItPy 준비 완료 — 아래 셀들을 하나씩 Shift+Enter


[INFO] [1784806528.906169035] [moveit_2678783690.moveit.py.cpp_initializer]: Initialize rclcpp
[INFO] [1784806528.906208394] [moveit_2678783690.moveit.py.cpp_initializer]: Initialize node parameters
[INFO] [1784806528.906216318] [moveit_2678783690.moveit.py.cpp_initializer]: Initialize node and executor
[INFO] [1784806528.909644480] [moveit_2678783690.moveit.py.cpp_initializer]: Spin separate thread
[INFO] [1784806528.912700232] [moveit_2678783690.moveit.ros.rdf_loader]: Loaded robot model in 0.00297886 seconds
[INFO] [1784806528.912720590] [moveit_2678783690.moveit.core.robot_model]: Loading robot model 'agx_arm'...
[INFO] [1784806528.912726879] [moveit_2678783690.moveit.core.robot_model]: No root/virtual joint specified in SRDF. Assuming fixed joint
[INFO] [1784806529.021731915] [moveit_2678783690.moveit.kinematics.kdl_kinematics_plugin]: Joint weights for group 'arm': 1 1 1 1 1 1
[INFO] [1784806529.455936022] [moveit_2678783690.moveit.ros.planning_scene_monitor]: Publishing maintain

In [2]:
def plan_and_execute(robot, component, logger,
                     single_plan_parameters=None,
                     multi_plan_parameters=None,
                     sleep_time=0.0):
    """계획 → 실행 헬퍼 (리포 ex03 과 동일)."""
    logger.info("계획 중...")
    if multi_plan_parameters is not None:
        result = component.plan(multi_plan_parameters=multi_plan_parameters)
    elif single_plan_parameters is not None:
        result = component.plan(single_plan_parameters=single_plan_parameters)
    else:
        result = component.plan()
    if result:
        logger.info("실행 중...")
        robot.execute(result.trajectory, controllers=[])  # controllers=[] → 자동 선택
    else:
        logger.error("계획 실패 (도달 불가/자기충돌?)")
    time.sleep(sleep_time)
    return bool(result)


def go_joints(joints, sleep_time=1.0):
    """관절공간 목표로 이동. 지정 안 한 관절은 0.0 으로 채운다."""
    arm.set_start_state_to_current_state()
    goal = RobotState(robot_model)
    positions = {f"joint{i}": 0.0 for i in range(1, 7)}
    positions.update({k: float(v) for k, v in joints.items()})
    goal.joint_positions = positions
    jc = construct_joint_constraint(
        robot_state=goal,
        joint_model_group=robot_model.get_joint_model_group("arm"),
    )
    arm.set_goal_state(motion_plan_constraints=[jc])
    return plan_and_execute(robot, arm, logger, sleep_time=sleep_time)


print("헬퍼 등록: plan_and_execute / go_joints")

헬퍼 등록: plan_and_execute / go_joints


## 모션 1 — SRDF 이름 상태 `home` (모든 관절 0, 곧게 선 자세)
공식 바인딩의 강점: SRDF 에 정의된 named state 로 바로 계획한다.

In [3]:
arm.set_start_state_to_current_state()
arm.set_goal_state(configuration_name="home")
plan_and_execute(robot, arm, logger, sleep_time=1.0)

[INFO] [1784806533.220822225] [nb01_arm_motions]: 계획 중...
[WARN] [1784806533.220857202] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784806533.220981575] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784806533.221036418] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784806533.221039923] [moveit_2678783690.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784806533.221048593] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784806533.221059827] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784806533.221128962] [moveit_2678783690.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will u

True

## 모션 2 — 관절공간 목표 여러 개
`go_joints({...})` 로 관절 각도를 직접 준다. 아래 값들을 바꿔가며 실험해봐. (자기충돌하는 조합은 계획이 실패하니 로그를 보면 됨.)

### 2a — 왼쪽 앞으로 뻗기

In [4]:
go_joints({"joint1": 0.6, "joint2": 0.4, "joint3": -0.5})

[INFO] [1784806536.615786038] [nb01_arm_motions]: 계획 중...
[WARN] [1784806536.615814061] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784806536.615897041] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784806536.615920037] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784806536.615922767] [moveit_2678783690.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784806536.615928793] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784806536.615936258] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784806536.615973704] [moveit_2678783690.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will u

True

### 2b — 오른쪽 앞으로 뻗기 (좌우 대칭)

In [5]:
go_joints({"joint1": -0.6, "joint2": 0.4, "joint3": -0.5})

[INFO] [1784806539.202715012] [nb01_arm_motions]: 계획 중...
[WARN] [1784806539.202744120] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784806539.202850557] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784806539.202872720] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784806539.202875245] [moveit_2678783690.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784806539.202881922] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784806539.202888763] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784806539.202922108] [moveit_2678783690.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will u

True

### 2c — 앞으로 숙이기 (어깨/팔꿈치 큰 굽힘)

In [6]:
go_joints({"joint2": 0.7, "joint3": -0.9})

[INFO] [1784806542.498225071] [nb01_arm_motions]: 계획 중...
[WARN] [1784806542.498264917] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784806542.498384820] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784806542.498405839] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784806542.498408489] [moveit_2678783690.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784806542.498415354] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784806542.498438238] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784806542.498479330] [moveit_2678783690.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will u

True

### 2d — 손목 비틀기 (joint4 손목 롤 사용)

In [7]:
go_joints({"joint1": -0.5, "joint2": 0.3, "joint4": 0.5})

[INFO] [1784806545.088291630] [nb01_arm_motions]: 계획 중...
[WARN] [1784806545.088341072] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784806545.088516977] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784806545.088557430] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784806545.088566118] [moveit_2678783690.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784806545.088585606] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784806545.088607491] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784806545.088672278] [moveit_2678783690.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will u

True

## 모션 3 — 포즈/IK 목표 (end-effector 위치·자세)
`tcp_link` 를 목표 pose 로 보내면 MoveIt 이 IK 를 풀어 관절해를 찾는다.

**권장 방법 — FK 로 도달 보장 포즈 만들기:** 아무 관절자세나 잡고 그 자세의 tcp_link
위치·자세(FK)를 목표로 쓰면 IK 해가 반드시 존재한다(그 관절자세 자체가 해니까).
그래서 '적당히 찍은 xyz 가 도달 불가라 계획 실패' 를 원천 차단한다.

In [8]:
from scipy.spatial.transform import Rotation

# 1) 씨앗 관절자세 → FK 로 tcp_link 변환행렬(4x4) 얻기
seed = RobotState(robot_model)
seed.joint_positions = {"joint1": 0.2, "joint2": 0.5, "joint3": -0.6,
                        "joint4": 0.0, "joint5": 0.5, "joint6": 0.0}
seed.update()
tf = np.asarray(seed.get_global_link_transform("tcp_link"))  # 4x4, 도달 보장
pos = tf[:3, 3]
quat = Rotation.from_matrix(tf[:3, :3]).as_quat()  # [x, y, z, w] (scipy 기본 순서)

# 2) PoseStamped 로 목표 지정 후 IK 계획
pose = PoseStamped()
pose.header.frame_id = "base_link"
pose.pose.position.x, pose.pose.position.y, pose.pose.position.z = map(float, pos)
(pose.pose.orientation.x, pose.pose.orientation.y,
 pose.pose.orientation.z, pose.pose.orientation.w) = map(float, quat)

arm.set_start_state_to_current_state()
arm.set_goal_state(pose_stamped_msg=pose, pose_link="tcp_link")
print("목표 pos:", np.round(pos, 4).tolist(), " quat:", np.round(quat, 4).tolist())
plan_and_execute(robot, arm, logger, sleep_time=1.0)

목표 pos: [0.0975, 0.0198, 0.3325]  quat: [-0.0807, 0.8046, 0.0587, 0.5854]


[INFO] [1784806548.129737381] [nb01_arm_motions]: 계획 중...
[WARN] [1784806548.129777061] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784806548.129917138] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784806548.129974972] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784806548.130005177] [moveit_2678783690.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784806548.130025866] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784806548.130046766] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784806548.130125361] [moveit_2678783690.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will u

True

### 3b — 리터럴 포즈 (리포 ex03 의 도달 보장 값, 대조용)
FK 없이 직접 좌표를 줄 수도 있다. 이 값은 리포에서 도달 확인된 포즈다.

In [19]:
pose = PoseStamped()
pose.header.frame_id = "base_link"
pose.pose.position.x = 0.1158
pose.pose.position.y = 0.0
pose.pose.position.z = 0.3133
pose.pose.orientation.x = 0.0
pose.pose.orientation.y = 0.8633
pose.pose.orientation.z = 0.0
pose.pose.orientation.w = 0.5047
arm.set_start_state_to_current_state()
arm.set_goal_state(pose_stamped_msg=pose, pose_link="tcp_link")
plan_and_execute(robot, arm, logger, sleep_time=1.0)

[INFO] [1784806639.472136835] [nb01_arm_motions]: 계획 중...
[WARN] [1784806639.472188021] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784806639.472369862] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784806639.472419074] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784806639.472428646] [moveit_2678783690.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784806639.472457543] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784806639.472480436] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784806639.472544563] [moveit_2678783690.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will u

True

## 모션 4 — 속도/가속 스케일링 (같은 동작을 느리게 vs 빠르게)
`PlanRequestParameters(robot, "ompl_rrtc")` 의 스케일링 팩터로 실행 속도를 조절한다.
(0.0~1.0. 실물 로봇 첫 구동 땐 0.05~0.1 처럼 낮게 시작하는 게 안전.)

In [10]:
def go_scaled(joints, vel, acc, sleep_time=1.0):
    arm.set_start_state_to_current_state()
    goal = RobotState(robot_model)
    positions = {f"joint{i}": 0.0 for i in range(1, 7)}
    positions.update({k: float(v) for k, v in joints.items()})
    goal.joint_positions = positions
    jc = construct_joint_constraint(
        robot_state=goal,
        joint_model_group=robot_model.get_joint_model_group("arm"),
    )
    arm.set_goal_state(motion_plan_constraints=[jc])
    prp = PlanRequestParameters(robot, "ompl_rrtc")
    prp.max_velocity_scaling_factor = vel
    prp.max_acceleration_scaling_factor = acc
    return plan_and_execute(robot, arm, logger,
                            single_plan_parameters=prp, sleep_time=sleep_time)

# 느릿느릿(5%) 로 뻗었다가
go_scaled({"joint1": 0.6, "joint2": 0.4, "joint3": -0.5}, 0.05, 0.05)
# 빠르게(90%) home 복귀
go_scaled({}, 0.9, 0.9)

[WARN] [1784806557.484659258] [moveit_py]: Parameter 'ompl_rrtc.plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784806557.487427856] [nb01_arm_motions]: 계획 중...
[INFO] [1784806557.487600213] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784806557.487641373] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784806557.487649766] [moveit_2678783690.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784806557.487668002] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784806557.487688544] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784806557.487748456] [moveit_2678783690.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefaul

True

## 모션 5 — 홈 복귀 (마무리)

In [11]:
arm.set_start_state_to_current_state()
arm.set_goal_state(configuration_name="home")
plan_and_execute(robot, arm, logger, sleep_time=1.0)

[INFO] [1784806563.867874091] [nb01_arm_motions]: 계획 중...
[WARN] [1784806563.867911092] [moveit_py]: Parameter 'plan_request_params.planning_time' not found in config use default value instead, check parameter type and namespace in YAML file
[INFO] [1784806563.868043837] [moveit_py]: Calling PlanningRequestAdapter 'ResolveConstraintFrames'
[INFO] [1784806563.868074843] [moveit_py]: Calling PlanningRequestAdapter 'ValidateWorkspaceBounds'
[WARN] [1784806563.868077892] [moveit_2678783690.moveit.ros.validate_workspace_bounds]: It looks like the planning volume was not specified. Using default values.
[INFO] [1784806563.868087123] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateBounds'
[INFO] [1784806563.868098331] [moveit_py]: Calling PlanningRequestAdapter 'CheckStartStateCollision'
[WARN] [1784806563.868153068] [moveit_2678783690.moveit.planners.ompl.planning_context_manager]: Cannot find planning configuration for group 'arm' using planner 'RRTConnectkConfigDefault'. Will u

True

## 반복하는 법 (핵심)
- 위 아무 셀이나 값 고치고 **Shift+Enter** → 팔이 즉시 그 목표로 움직인다.
  colcon 재빌드도, 런치 재시작도 없다. 커널이 살아있고 move_group·컨트롤러가 계속 떠 있으니까.
- **설정 셀(`MoveItPy(...)`)은 다시 실행하지 말 것** — 커널당 한 번.
- 끝낼 땐 커널을 종료하면 되고, 그때 "Kernel died" 가 떠도 정상이다
  (MoveItPy 종료 시 알려진 무해한 소멸자 이슈).
- URDF/SRDF/컨트롤러 설정을 바꿀 때만 런치를 Ctrl-C 후 다시 띄우면 된다.